# Miscellaneous Experiments

In [ ]:
import torch
import triton
import triton.language as tl

from kernels.fma_kernel import fma_triton, fma_kernel_runtime_stride, fma_kernel
from benchmarks.util import save_ptx

## Using a non power of 2 as block size

In [3]:
x = torch.randn(1_000_000, device='cuda')
y = torch.randn(1_000_000, device='cuda')
z = torch.randn(1_000_000, device='cuda')
out = torch.empty_like(x)

try:
    fma_kernel[(triton.cdiv(x.numel(), 1000),)](x, y, z, out, x.numel(), BLOCK_SIZE=1000)
except Exception as e:
    print(type(e).__name__)
    print(str(e)[-300:])

CompilationError
at 3:33:
def fma_kernel(x_ptr, y_ptr, z_ptr, out_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
                                 ^
arange's range must be a power of 2


## After handling the above issue

In [4]:
out_triton = fma_triton(x, y, z)
out_triton_runtime_stride = torch.empty_like(x)
fma_kernel_runtime_stride[(triton.cdiv(x.numel(), 1024),)](
    x, y, z, out_triton_runtime_stride, x.numel(), 1024, BLOCK_SIZE=1024)

diff = (out_triton - out_triton_runtime_stride).abs()
print(f"Max absolute difference: {diff.max().item()}")
print(torch.allclose(out_triton, out_triton_runtime_stride, atol=1e-6))

Max absolute difference: 0.0
True


In [7]:
compiled = fma_kernel_runtime_stride[(triton.cdiv(x.numel(), 1024),)](
    x, y, z, out, x.numel(), 1024, BLOCK_SIZE=1024)
save_ptx(compiled, 'ptx_analysis/fma_runtime_stride.ptx')

PTX saved to ptx_analysis/ptx_analysis/fma_runtime_stride.ptx


## Comparing the ptx generated at both scenerios

In [8]:
!grep -nE "shl|mul\.(lo|wide)|mad" ptx_analysis/fma_runtime_stride.ptx

40:	shl.b32 	%r35, %r34, 2;
45:	mad.lo.s32 	%r39, %r38, %r33, %r37;
51:	mul.wide.s32 	%rd13, %r39, 4;
53:	mul.wide.s32 	%rd14, %r40, 4;


In [9]:
!grep -nE "shl|mul\.(lo|wide)|mad" ptx_analysis/fma_kernel.ptx

36:	shl.b32 	%r34, %r33, 10;
41:	shl.b32 	%r36, %r35, 2;
51:	mul.wide.s32 	%rd13, %r39, 4;


## Practical Bandwidth Analysis of Triton Kernels

In [3]:
for n_cols in [1024, 4096, 8192]:
    x = torch.randn(4096, n_cols, device='cuda')
    ms = triton.testing.do_bench(lambda: x.clone())
    gb = 2 * x.numel() * x.element_size() / 1e9
    print(n_cols, f"{ms:.4f} ms", f"{gb / (ms * 1e-3):.1f} GB/s")

1024 0.1423 ms 235.9 GB/s
4096 0.5569 ms 241.0 GB/s
8192 1.1142 ms 240.9 GB/s
